In [73]:
import pandas as pd 
import numpy as np 
from dotenv import load_dotenv
import os

load_dotenv(r'C:\Users\Utilizador\Desktop\CDPROJ2\Cinecia_Dados_PROJ2\.env') 

DATA_PATH = os.getenv('CSV_PATH')
STATIONS_PATH = os.getenv('STATIONS_PATH')

In [64]:
VALUE_COLS = [f'value{i}' for i in range(1, 32)]
USE_COLS = ['id', 'year', 'month', 'element'] + VALUE_COLS

DTYPES = {
    'id': 'string',
    'year': 'int16',
    'month': 'int8',
    'element': 'string',
}
for col in VALUE_COLS:
    DTYPES[col] = 'float32'

CHUNK_SIZE = 500_000

df = pd.read_csv(DATA_PATH, usecols=USE_COLS, dtype=DTYPES, na_values=[-9999], chunksize=CHUNK_SIZE)

chunk = next(df)

print(f"Shape of the chunk: {chunk.shape}")
print(f"Data types of the chunk:\n{chunk.dtypes}")
print(f"First 5 rows of the chunk:\n{chunk.head()}")

Shape of the chunk: (500000, 35)
Data types of the chunk:
id          string
year         int16
month         int8
element     string
value1     float32
value2     float32
value3     float32
value4     float32
value5     float32
value6     float32
value7     float32
value8     float32
value9     float32
value10    float32
value11    float32
value12    float32
value13    float32
value14    float32
value15    float32
value16    float32
value17    float32
value18    float32
value19    float32
value20    float32
value21    float32
value22    float32
value23    float32
value24    float32
value25    float32
value26    float32
value27    float32
value28    float32
value29    float32
value30    float32
value31    float32
dtype: object
First 5 rows of the chunk:
            id  year  month element  value1  value2  value3  value4  value5  \
0  ACW00011604  1949      1    TMAX   289.0   289.0   283.0   283.0   289.0   
1  ACW00011604  1949      2    TMAX   267.0   278.0   272.0   267.0   278.0   

## Percentagem de NULL

In [65]:
null_pct = (chunk.isnull().sum() / len(chunk) * 100).round(2)
null_df = null_pct.reset_index()
null_df.columns = ['coluna', 'pct_nulos (%)']

print("Percentagem de valores nulos por coluna:")
null_df

Percentagem de valores nulos por coluna:


,coluna,pct_nulos (%)
0,id,0.00
1,year,0.00
2,month,0.00
3,element,0.00
4,value1,9.96
5,value2,10.00
6,value3,9.95
7,value4,10.08
8,value5,9.94
9,value6,9.95


## Ano mais antigo

In [66]:
# Para processar todo o ficheiro por chunks e acumular min/max
year_agg = {}  # {id: {'min': ..., 'max': ...}}

reader = pd.read_csv(
    DATA_PATH,
    usecols=['id', 'year'],
    dtype={'id': 'string', 'year': 'int16'},
    chunksize=CHUNK_SIZE
)

for ck in reader:
    grp = ck.groupby('id')['year'].agg(['min', 'max'])
    for station_id, row in grp.iterrows():
        if station_id not in year_agg:
            year_agg[station_id] = {'min': row['min'], 'max': row['max']}
        else:
            year_agg[station_id]['min'] = min(year_agg[station_id]['min'], row['min'])
            year_agg[station_id]['max'] = max(year_agg[station_id]['max'], row['max'])

station_years = pd.DataFrame.from_dict(year_agg, orient='index')
station_years.index.name = 'id'
station_years.columns = ['ano_mais_antigo', 'ano_mais_recente']
station_years = station_years.reset_index()

print(f"Total de estações: {len(station_years)}")
station_years.head(10)

Total de estações: 40133


,id,ano_mais_antigo,ano_mais_recente
0,ACW00011604,1949,1949
1,ACW00011647,1961,1961
2,AE000041196,1944,2019
3,AEM00041194,1983,2019
4,AEM00041217,1983,2019
5,AEM00041218,1994,2019
6,AF000040930,1973,1992
7,AFM00040938,1973,2019
8,AFM00040948,1966,2019
9,AFM00040990,1973,2019


## Temperatura Media

In [68]:
# Calcular média ignorando NaN apenas sobre as colunas value1..value31
chunk['daily_avg_temp'] = chunk[VALUE_COLS].mean(axis=1)

print("Primeiras linhas com daily_avg_temp:")
chunk[['id', 'year', 'month', 'element', 'daily_avg_temp']].head(10)

Primeiras linhas com daily_avg_temp:


,id,year,month,element,daily_avg_temp
0,ACW00011604,1949,1,TMAX,274.612915
1,ACW00011604,1949,2,TMAX,271.142853
2,ACW00011604,1949,3,TMAX,277.935486
3,ACW00011604,1949,4,TMAX,287.166656
4,ACW00011604,1949,5,TMAX,291.354828
5,ACW00011604,1949,6,TMAX,294.833344
6,ACW00011604,1949,7,TMAX,298.709686
7,ACW00011647,1961,10,TMAX,272.000000
8,AE000041196,1944,3,TMAX,323.166656
9,AE000041196,1944,4,TMAX,321.466675


## Grupos 


In [69]:
temp_by_station_year = (
    chunk
    .groupby(['id', 'year'])['daily_avg_temp']
    .mean()
    .round(2)
    .reset_index()
)

print("Temperatura média anual por estação:")
temp_by_station_year.head(15)

Temperatura média anual por estação:


,id,year,daily_avg_temp
0,ACW00011604,1949,285.109985
1,ACW00011647,1961,272.000000
2,AE000041196,1944,348.869995
3,AE000041196,1945,318.230011
4,AE000041196,1955,317.920013
5,AE000041196,1956,318.109985
6,AE000041196,1957,311.390015
7,AE000041196,1958,317.899994
8,AE000041196,1959,309.899994
9,AE000041196,1960,316.929993


## Filtrar as 5 estações

In [ ]:
# Ler o ficheiro de estações (largura fixa)
# Formato: ID (0-11), LAT (12-20), LON (21-30), ELEV (31-37), STATE (38-40),
#          NAME (41-71), GSFLAG (72-75), HCNFLAG (76-79), WMOID (80-85)
stations_path = STATIONS_PATH
if not os.path.exists(stations_path):
    alt_path = os.path.splitext(stations_path)[0] + '.txt'
    if os.path.exists(alt_path):
        stations_path = alt_path
    else:
        raise FileNotFoundError(
            f"Stations file not found: {stations_path!r}. "
            f"Verify STATIONS_PATH or put the file in the data folder."
        )

stations = pd.read_fwf(
    stations_path,
    colspecs=[(0, 11), (12, 20), (21, 30), (31, 37), (38, 40), (41, 71)],
    names=['id', 'lat', 'lon', 'elev', 'state', 'name'],
    dtype={'id': 'string', 'name': 'string'}
)
stations['name'] = stations['name'].str.strip()

# IDs das 5 estações portuguesas (verificar no ficheiro de estações)
PT_NAMES = ['HORTA', 'FUNCHAL', 'LISBOA', 'CASTELO BRANCO', 'FARO']

pt_stations = stations[stations['name'].str.upper().isin(PT_NAMES)]
print("Estações portuguesas encontradas:")
print(pt_stations[['id', 'name']])

PT_IDS = pt_stations['id'].tolist()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Utilizador\\Desktop\\CDPROJ2\\Cinecia_Dados_PROJ2\\data\\ghcnd_stations.csv'